# R(2+1)D-18 RD: frozen evaluation on the reused official test split

**Do not run until the team has frozen all model choices.** The embedded freeze
(`configs/r2plus1d18_rd_001_evaluation.json`) must have `team_freeze_confirmed: true`
from a reviewed commit; otherwise this notebook stops before opening any test data.

Attach (read-only) the final training output `r2plus1d18-rd-001-train-04` and the
saved U-Net++ test-input output `unetpp-rd-002-test-inputs`. The notebook verifies
the training artifact (config digest, run fingerprint, 50/50 epochs, selected
**epoch 15**, checkpoint SHA256) *before* reading `test.csv`, checks the GPU runtime
equals training, re-verifies all 1,077 cached test tensors, then runs one forward
pass per clip at **threshold 0.5**. No optimizer, retraining, threshold search,
calibration or epoch selection. Inference timing is recorded.

The official test split was already used for the U-Net++ result, so report this as
a **reused official test benchmark**. Clip-level splits may share patients; this
educational result is not a clinical or unseen-patient validation.


In [ ]:
import gc
import hashlib
import json
import sys
import time
import types
import zipfile
from pathlib import Path
import cv2
import numpy as np
import pandas as pd
import sklearn
import torch
import torchvision
import huggingface_hub
from IPython.display import display

if not Path('/kaggle/working').is_dir():
    raise RuntimeError('Submit this notebook to Kaggle; its remote environment runs the workload.')
torch.set_num_threads(2)
cv2.setNumThreads(1)
torch.backends.cudnn.benchmark = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
r2p1_runtime = {'python': sys.version, 'torch': str(torch.__version__), 'torchvision': str(torchvision.__version__),
    'numpy': np.__version__, 'opencv': cv2.__version__, 'pandas': pd.__version__, 'sklearn': sklearn.__version__,
    'huggingface_hub': huggingface_hub.__version__, 'cuda': torch.version.cuda,
    'cudnn': torch.backends.cudnn.version(), 'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'gpu_total_memory_bytes': torch.cuda.get_device_properties(0).total_memory if torch.cuda.is_available() else None,
    'cpu_torch_threads': torch.get_num_threads(), 'cudnn_benchmark': False,
    'matmul_tf32': False, 'cudnn_tf32': False}
print(r2p1_runtime)


In [ ]:
R2P1_FREEZE = {'schema_version': 1,
 'evaluation_id': 'r2plus1d18_rd_001_test',
 'model_label': 'R(2+1)D-18',
 'frozen_at_utc': '2026-11-13T00:00:00+00:00',
 'team_freeze_confirmed': True,
 'team_freeze_note': 'Confirmed 2026-11-13 by the R(2+1)D-18 owner (teammate 1), who instructed '
                     'the single test run. This fixes the R(2+1)D-18 choices only (epoch 15, '
                     'threshold 0.5); a team-wide freeze of all models is not recorded here.',
 'training_experiment_id': 'r2plus1d18_rd_001_amp',
 'training_source': 'namanbabariya/r2plus1d18-rd-001-train-04',
 'training_source_version': 1,
 'training_epochs_completed': 50,
 'selected_epoch': 15,
 'selection_rule': 'highest validation accuracy; earliest on ties',
 'checkpoint_file': 'best.pt',
 'checkpoint_sha256': 'c041f7e2f635e9f3203e76045fbad247c3243788d527425d27cc73b27b93ec16',
 'training_config_digest': 'afecb01905e0ac77ede0728c4b00149652f18184a4fc14a691acb93f729dbfa4',
 'training_run_fingerprint': '1103636b032e120507f278df8f50ba763b6c8feb4fb5d10f3b6941be1ba68902',
 'training_implementation_sha256': {'unetpp_baseline': '5180b7022ecd54f3319fa0d780a992c95fa404a5ce8dbae15766db24eef3c5ed',
                                    'unetpp_training': '7bc8a073d45e08a4c6d24595508be65f9596dbc50500f6dc6799bdc23d7fe2e1',
                                    'unetpp_data': 'e6e186ed51bd5b33260e09b7b7b64caea23f28103bbdcd3749125b44c30feb61',
                                    'unetpp_kaggle': 'fc39badd5f9e20377e3c3052200bf188df8e9c98ccd6a5ef5a1278cf14c5013d',
                                    'video_cache': '83ce03c382213cfe1648299a8385c5be12f1cb6fd1844ea1e3124abf55f2957b',
                                    'comparison_export': '0f7d133fc466c93abe87dd670dbe1b2351b5c38cf82b5547cbc441190e916747',
                                    'r2plus1d18_baseline': 'b47554479e9142464d8b968ce7b6446d58a9f2a50b5f063b2660ec3df7769f47'},
 'training_runtime': {'python': '3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]',
                      'torch': '2.11.0+cu130',
                      'torchvision': '0.26.0+cu130',
                      'numpy': '2.1.3',
                      'opencv': '4.14.0',
                      'pandas': '2.3.3',
                      'sklearn': '1.6.1',
                      'huggingface_hub': '1.33.0',
                      'cuda': '13.0',
                      'cudnn': 92700,
                      'gpu': 'Tesla T4',
                      'gpu_total_memory_bytes': 15636037632,
                      'cpu_torch_threads': 2,
                      'cudnn_benchmark': False,
                      'matmul_tf32': False,
                      'cudnn_tf32': False},
 'dataset_revision': '7867e2504ed3845a46ff7790a2f8c3ffcdc08947',
 'task': 'non_rd_vs_rd',
 'positive_class': 'rd',
 'threshold': 0.5,
 'precision': '16-mixed',
 'physical_batch': 4,
 'preprocessing_identity': {'preprocessing': {'num_frames': 96,
                                              'height': 128,
                                              'width': 128,
                                              'grayscale': 'arithmetic channel mean',
                                              'padding': 'symmetric zero padding to square',
                                              'resize': 'full time/spatial trilinear interpolation',
                                              'align_corners': False,
                                              'normalization': 'divide by 255',
                                              'dtype': 'float32',
                                              'layout': 'B,C,D,H,W',
                                              'frame_subsampling': False,
                                              'augmentation': 'none',
                                              'roundoff_tolerance': 1e-06},
                            'implementation': '5180b7022ecd54f3319fa0d780a992c95fa404a5ce8dbae15766db24eef3c5ed',
                            'torch': '2.11.0+cpu',
                            'opencv': '4.14.0',
                            'numpy': '2.1.3'},
 'test_cache': {'source': 'bijaykhadka1710/unetpp-rd-002-test-inputs',
                'source_version': 1,
                'preparation_evaluation_id': 'unetpp_rd_002_test',
                'preparation_freeze_digest': '33c392efdd3cc0bb34c799a692f0411aa1711970e25c31ca49fc9d76db652b49'},
 'test_clips': 1077,
 'test_label_counts': {'0': 977, '1': 100},
 'test_policy': 'single fixed-checkpoint evaluation; no optimizer, threshold search, epoch '
                'selection or retraining',
 'test_accessed_at_freeze': False,
 'benchmark_label': 'reused official test benchmark',
 'reference_test_rows': [{'source': 'U-Net++ epoch 48 '
                                    '(bijaykhadka1710/unetpp-rd-002-test-evaluation/1, verified)',
                          'partition': 'reused official test benchmark',
                          'TP': 86,
                          'TN': 964,
                          'FP': 13,
                          'FN': 14,
                          'sensitivity': 0.86,
                          'specificity': 0.986694,
                          'precision': 0.868687,
                          'F1': 0.864322,
                          'accuracy': 0.97493,
                          'balanced_accuracy': 0.923347,
                          'average_precision': 0.926845,
                          'ROC_AUC': 0.986858}],
 'disclosures': ['The official test split was already exposed at project level by the U-Net++ '
                 'evaluation; this is a reused official test benchmark, not an untouched holdout.',
                 'Reuses the U-Net++ test-input cache (pinned preparation freeze digest); its '
                 'preprocessing identity must equal the training cache identity.',
                 'Epoch 15 and threshold 0.5 were fixed on validation before any R(2+1)D-18 test '
                 'access; no test-driven tuning.',
                 'FP16 autocast inference on one Kaggle T4 GPU, physical batch 4; timing is '
                 'recorded with that hardware/batch/precision.',
                 'Official splits are clip-level with possible same-patient overlap; one seed; no '
                 'unseen-patient or clinical-validation claim.',
                 "Inherited U-Net++ recipe; not R(2+1)D-18's tuned best. torchvision R(2+1)D-18 is "
                 'not an ERDES Table 8 row.']}


In [ ]:
# In-memory package so the notebook needs no repository upload.
r2p1_package = types.ModuleType('rd_ultrasound_ml')
r2p1_package.__path__ = []
sys.modules['rd_ultrasound_ml'] = r2p1_package
r2p1_sources_embedded = {}
def r2p1_load_module(name, source):
    module = types.ModuleType('rd_ultrasound_ml.' + name)
    module.__file__ = '<embedded ' + name + '.py>'
    sys.modules[module.__name__] = module
    exec(compile(source, module.__file__, 'exec'), module.__dict__)
    r2p1_sources_embedded[name] = source


In [ ]:
R2P1_MODULE_SOURCE = '"""Independent implementation of the released ERDES UNet++ classifier protocol.\n\nArchitecture/protocol reference (read-only; no upstream source imported):\nhttps://github.com/OSUPCVLab/ERDES/tree/4af67e0b9f4667edbae75a64a819884ee14d7a3b\nThe authors\' encoder-only classifier uses MONAI TwoConv/Down blocks. The\nequivalent operations below use PyTorch directly; this is not a full nested\nsegmentation UNet++. See docs/research/erdes-unetpp-protocol.md for\nattribution, numerical parity checks, and paper/code discrepancies.\n"""\nfrom pathlib import Path\nimport time\n\nimport cv2\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass UNetPPRDClassifier(nn.Module):\n    """Five double-convolution blocks, four pools, global pool and 256→128→1."""\n\n    def __init__(self):\n        super().__init__()\n        widths = (32, 32, 64, 128, 256)\n        blocks = []\n        incoming = 1\n        for index, outgoing in enumerate(widths):\n            layers = [] if index == 0 else [nn.MaxPool3d(2, stride=2)]\n            for channels_in in (incoming, outgoing):\n                layers.extend([\n                    nn.Conv3d(channels_in, outgoing, kernel_size=3, stride=1, padding=1, bias=True),\n                    nn.InstanceNorm3d(outgoing, eps=1e-5, momentum=0.1, affine=True, track_running_stats=False),\n                    nn.LeakyReLU(negative_slope=0.1, inplace=True),\n                ])\n            blocks.append(nn.Sequential(*layers))\n            incoming = outgoing\n        self.encoder = nn.Sequential(*blocks)\n        self.pool = nn.AdaptiveAvgPool3d(1)\n        self.head = nn.Sequential(nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1))\n\n    def forward(self, inputs):\n        features = self.encoder(inputs)\n        return self.head(self.pool(features).flatten(1)).squeeze(1)\n\n\ndef resize_grayscale_volume(gray_frames, target=(96, 128, 128)):\n    """Full-time-axis trilinear interpolation, not uniform frame subsampling."""\n    if not isinstance(gray_frames, np.ndarray) or gray_frames.ndim != 3:\n        raise ValueError(\'Expected a D,H,W numpy array of decoded grayscale values\')\n    if min(gray_frames.shape) <= 0 or len(target) != 3 or min(target) <= 0:\n        raise ValueError(\'Source and target dimensions must be positive\')\n    if not np.isfinite(gray_frames).all() or gray_frames.min() < 0 or gray_frames.max() > 255:\n        raise ValueError(\'Decoded grayscale values must be finite and in [0,255]\')\n    tensor = torch.from_numpy(np.ascontiguousarray(gray_frames, dtype=np.float32))[None]\n    height, width = tensor.shape[-2:]\n    difference = abs(height - width)\n    if difference:\n        before, after = difference // 2, difference - difference // 2\n        padding = (0, 0, before, after) if height < width else (before, after, 0, 0)\n        tensor = F.pad(tensor, padding, mode=\'constant\', value=0)\n    tensor = F.interpolate(tensor[None], size=tuple(target), mode=\'trilinear\', align_corners=False)[0]\n    tensor.div_(255.)\n    minimum, maximum = tensor.min().item(), tensor.max().item()\n    if not torch.isfinite(tensor).all() or minimum < -1e-6 or maximum > 1 + 1e-6:\n        raise ValueError(f\'Unexpected preprocessed range: {minimum}, {maximum}\')\n    return tensor.clamp_(0., 1.)\n\n\ndef prepare_unetpp_clip(path, expected_frames, expected_height=None, expected_width=None, target=(96, 128, 128)):\n    """Decode all frames into one preallocated grayscale buffer, with metadata checks."""\n    path = Path(path)\n    if not path.is_file():\n        raise FileNotFoundError(path)\n    if expected_frames < 1:\n        raise ValueError(\'Expected frame count must be positive\')\n    capture = cv2.VideoCapture(str(path))\n    decoded = 0\n    gray = None\n    try:\n        if not capture.isOpened():\n            raise ValueError(f\'Cannot decode {path.name}\')\n        while True:\n            ok, frame = capture.read()\n            if not ok:\n                break\n            if frame.dtype != np.uint8 or frame.ndim != 3 or frame.shape[2] != 3:\n                raise ValueError(f\'{path.name}: expected uint8 three-channel decoded frame\')\n            height, width = frame.shape[:2]\n            if ((expected_height is not None and height != expected_height)\n                    or (expected_width is not None and width != expected_width)):\n                raise ValueError(f\'{path.name}: decoded dimensions differ from metadata\')\n            if decoded >= expected_frames:\n                raise ValueError(f\'{path.name}: decoded more frames than metadata expects\')\n            if gray is None:\n                gray = np.empty((expected_frames, height, width), dtype=np.float32)\n            if frame.shape[:2] != gray.shape[1:]:\n                raise ValueError(f\'{path.name}: changing frame dimensions\')\n            # Arithmetic channel mean is invariant to OpenCV\'s BGR channel order.\n            gray[decoded] = frame.mean(axis=2, dtype=np.float32)\n            decoded += 1\n    finally:\n        capture.release()\n    if decoded != expected_frames:\n        raise ValueError(f\'{path.name}: decoded {decoded} frames; metadata expects {expected_frames}\')\n    return resize_grayscale_volume(gray, target)\n\n\ndef benchmark_cuda_batch(model_factory, inputs, labels, physical_batch, warmup=1, steps=3):\n    """Disposable FP32 training/inference probe; never produces a research checkpoint."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'A CUDA runtime is required for the hardware probe\')\n    if physical_batch < 1 or steps < 1 or warmup < 1:\n        raise ValueError(\'Positive batch size, warmup and measurement steps are required\')\n    device = torch.device(\'cuda\')\n    model = optimizer = x = y = logits = loss = None\n    initial_allocated = torch.cuda.memory_allocated()\n    torch.cuda.reset_peak_memory_stats()\n    try:\n        model = model_factory().to(device)\n        optimizer = torch.optim.Adam(model.parameters(), lr=1.5e-5, weight_decay=0.)\n        indices = torch.arange(physical_batch) % len(inputs)\n        x, y = inputs[indices].to(device), labels[indices].to(device)\n        loss_function = nn.BCEWithLogitsLoss()\n        train_seconds = []\n        for iteration in range(warmup + steps):\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            torch.cuda.synchronize()\n            start = time.perf_counter()\n            logits = model(x)\n            loss = loss_function(logits, y)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite hardware-probe loss\')\n            loss.backward()\n            if iteration == 0:\n                gradients = [parameter.grad for parameter in model.parameters()]\n                if not all(g is not None and torch.isfinite(g).all() for g in gradients):\n                    raise ValueError(\'Nonfinite or missing hardware-probe gradients\')\n                if not any(torch.count_nonzero(g).item() for g in gradients):\n                    raise ValueError(\'All hardware-probe gradients are zero\')\n                del gradients\n            optimizer.step()\n            torch.cuda.synchronize()\n            elapsed = time.perf_counter() - start\n            if iteration >= warmup:\n                train_seconds.append(elapsed)\n        model.eval()\n        inference_seconds = []\n        with torch.no_grad():\n            for _ in range(steps):\n                torch.cuda.synchronize()\n                start = time.perf_counter()\n                model(x)\n                torch.cuda.synchronize()\n                inference_seconds.append(time.perf_counter() - start)\n        return {\'physical_batch\': physical_batch, \'status\': \'passed\', \'precision\': \'float32\',\n            \'training_step_seconds_median\': float(np.median(train_seconds)),\n            \'validation_step_seconds_median\': float(np.median(inference_seconds)),\n            \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(),\n            \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(),\n            \'initial_allocated_bytes\': initial_allocated,\n            \'finite_nonzero_gradients\': True, \'loss_last\': loss.item(),\n            \'weights_discarded\': True}\n    except torch.cuda.OutOfMemoryError:\n        return {\'physical_batch\': physical_batch, \'status\': \'out_of_memory\', \'precision\': \'float32\',\n            \'weights_discarded\': True}\n    finally:\n        del model, optimizer, x, y, logits, loss\n        torch.cuda.empty_cache()\n'
r2p1_load_module('unetpp_baseline', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""Sample-weighted accumulation and resumable UNet++ training infrastructure.\n\nOwn implementation; the ERDES reference supplies the task/model/protocol choices.\nCheckpoints contain plain containers and tensors and load with weights_only=True.\n"""\nfrom contextlib import nullcontext\nimport hashlib\nimport json\nfrom pathlib import Path\nimport random\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.utils.data import DataLoader\nfrom sklearn.metrics import average_precision_score, roc_auc_score\n\n\ndef json_digest(value):\n    content = json.dumps(value, sort_keys=True, separators=(\',\', \':\'), allow_nan=False).encode()\n    return hashlib.sha256(content).hexdigest()\n\n\ndef file_digest(path):\n    digest = hashlib.sha256()\n    with Path(path).open(\'rb\') as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b\'\'):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef write_json_atomic(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + \'.tmp\')\n    temporary.write_text(json.dumps(value, indent=2, allow_nan=False) + \'\\n\', encoding=\'utf-8\')\n    temporary.replace(path)\n\n\ndef seed_training(seed):\n    random.seed(seed)\n    np.random.seed(seed)\n    torch.manual_seed(seed)\n    if torch.cuda.is_available():\n        torch.cuda.manual_seed_all(seed)\n\n\ndef capture_random_state():\n    state = np.random.get_state()\n    return {\'python\': random.getstate(), \'numpy\': {\n        \'algorithm\': state[0], \'keys\': torch.from_numpy(state[1].astype(np.int64)),\n        \'position\': state[2], \'has_gaussian\': state[3], \'cached_gaussian\': state[4]},\n        \'torch\': torch.get_rng_state(),\n        \'cuda\': torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}\n\n\ndef restore_random_state(value):\n    random.setstate(value[\'python\'])\n    state = value[\'numpy\']\n    np.random.set_state((state[\'algorithm\'], state[\'keys\'].cpu().numpy().astype(np.uint32),\n                         state[\'position\'], state[\'has_gaussian\'], state[\'cached_gaussian\']))\n    torch.set_rng_state(value[\'torch\'].cpu())\n    if value[\'cuda\'] and torch.cuda.is_available():\n        torch.cuda.set_rng_state_all([item.cpu() for item in value[\'cuda\']])\n\n\ndef epoch_order(length, seed, epoch):\n    if length < 1 or epoch < 1:\n        raise ValueError(\'Dataset length and epoch must be positive\')\n    generator = torch.Generator().manual_seed(seed + epoch - 1)\n    return torch.randperm(length, generator=generator).tolist()\n\n\ndef autocast_for(device, precision):\n    device = torch.device(device)\n    if precision not in {\'float32\', \'16-mixed\'}:\n        raise ValueError(\'Expected float32 or 16-mixed precision\')\n    if precision == \'16-mixed\':\n        if device.type != \'cuda\':\n            raise ValueError(\'16-mixed training requires a CUDA device\')\n        return torch.autocast(device_type=\'cuda\', dtype=torch.float16)\n    return nullcontext()\n\n\ndef train_epoch_accumulated(model, dataset, order, optimizer, device, physical_batch,\n                            effective_batch, precision=\'float32\', scaler=None,\n                            start_position=0, loss_sum=0., examples_seen=0,\n                            optimizer_updates=0, skipped_updates=0, on_boundary=None):\n    """Resume only at clean optimizer boundaries; normalize the short final window."""\n    device = torch.device(device)\n    length = len(dataset)\n    if sorted(order) != list(range(length)):\n        raise ValueError(\'Epoch order must contain every dataset index exactly once\')\n    if physical_batch < 1 or effective_batch < 1 or effective_batch % physical_batch:\n        raise ValueError(\'Effective batch must be a positive multiple of physical batch\')\n    if not (0 <= start_position <= length) or (start_position % effective_batch and start_position != length):\n        raise ValueError(\'Resume position must be an optimizer-update boundary\')\n    if examples_seen != start_position:\n        raise ValueError(\'Saved epoch example count differs from its cursor\')\n    if precision == \'16-mixed\' and (scaler is None or not scaler.is_enabled()):\n        raise ValueError(\'Mixed precision requires an enabled GradScaler\')\n    model.train()\n    optimizer.zero_grad(set_to_none=True)\n    # An explicit generator avoids a new iterator consuming the model\'s global RNG on resume.\n    generator = torch.Generator().manual_seed(0)\n    loader = DataLoader(dataset, batch_size=physical_batch, sampler=order[start_position:],\n                        num_workers=0, pin_memory=device.type == \'cuda\', generator=generator)\n    position = start_position\n    window_start = start_position\n    loss_function = nn.BCEWithLogitsLoss()\n    for inputs, labels in loader:\n        inputs, labels = inputs.to(device, non_blocking=True), labels.to(device, non_blocking=True)\n        window_size = min(effective_batch, length - window_start)\n        with autocast_for(device, precision):\n            logits = model(inputs).reshape(-1)\n            loss = loss_function(logits, labels)\n        if not torch.isfinite(loss):\n            raise ValueError(\'Nonfinite training loss; resume from the last valid checkpoint\')\n        normalized = loss * (len(labels) / window_size)\n        if precision == \'16-mixed\':\n            scaler.scale(normalized).backward()\n        else:\n            normalized.backward()\n        loss_sum += loss.item() * len(labels)\n        position += len(labels)\n        examples_seen += len(labels)\n        if position == window_start + window_size:\n            if precision == \'16-mixed\':\n                # Keep one scale throughout the entire effective batch. Step/update only here.\n                old_scale = scaler.get_scale()\n                scaler.step(optimizer)\n                scaler.update()\n                skipped_updates += int(scaler.get_scale() < old_scale)\n            else:\n                if not all(p.grad is None or torch.isfinite(p.grad).all() for p in model.parameters()):\n                    raise ValueError(\'Nonfinite FP32 gradients; optimizer was not stepped\')\n                optimizer.step()\n            optimizer_updates += 1\n            optimizer.zero_grad(set_to_none=True)\n            progress = {\'position\': position, \'loss_sum\': loss_sum, \'examples_seen\': examples_seen,\n                        \'optimizer_updates\': optimizer_updates, \'skipped_updates\': skipped_updates}\n            if on_boundary is not None:\n                on_boundary(progress.copy())\n            window_start = position\n    return {\'position\': position, \'loss_sum\': loss_sum, \'examples_seen\': examples_seen,\n            \'optimizer_updates\': optimizer_updates, \'skipped_updates\': skipped_updates,\n            \'train_loss\': loss_sum / length}\n\n\ndef validate_model(model, dataset, device, physical_batch, precision=\'float32\'):\n    device = torch.device(device)\n    model.eval()\n    generator = torch.Generator().manual_seed(0)\n    loader = DataLoader(dataset, batch_size=physical_batch, shuffle=False, num_workers=0,\n                        pin_memory=device.type == \'cuda\', generator=generator)\n    loss_sum = 0.\n    labels, probabilities = [], []\n    criterion = nn.BCEWithLogitsLoss()\n    with torch.no_grad():\n        for inputs, target in loader:\n            inputs, target = inputs.to(device, non_blocking=True), target.to(device, non_blocking=True)\n            with autocast_for(device, precision):\n                logits = model(inputs).reshape(-1)\n                loss = criterion(logits, target)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite validation loss\')\n            loss_sum += loss.item() * len(target)\n            labels.extend(target.cpu().tolist())\n            probabilities.extend(torch.sigmoid(logits.float()).cpu().tolist())\n    return loss_sum / len(dataset), np.asarray(labels), np.asarray(probabilities)\n\n\ndef binary_metrics(labels, probabilities, threshold=.5):\n    labels, probabilities = np.asarray(labels), np.asarray(probabilities, dtype=np.float64)\n    if labels.ndim != 1 or len(labels) == 0 or labels.shape != probabilities.shape or not np.isin(labels, [0, 1]).all():\n        raise ValueError(\'Expected non-empty matching binary labels and probability vectors\')\n    if not np.isfinite(probabilities).all() or np.any((probabilities < 0) | (probabilities > 1)):\n        raise ValueError(\'Probabilities must be finite and in [0,1]\')\n    prediction = probabilities >= threshold\n    tp, tn = int(np.sum((labels == 1) & prediction)), int(np.sum((labels == 0) & ~prediction))\n    fp, fn = int(np.sum((labels == 0) & prediction)), int(np.sum((labels == 1) & ~prediction))\n    reasons = {}\n    def divide(name, numerator, denominator):\n        if denominator == 0:\n            reasons[name] = \'zero denominator\'\n            return None\n        return numerator / denominator\n    sensitivity = divide(\'sensitivity\', tp, tp + fn)\n    specificity = divide(\'specificity\', tn, tn + fp)\n    value = {\'TP\': tp, \'TN\': tn, \'FP\': fp, \'FN\': fn, \'sensitivity\': sensitivity,\n        \'specificity\': specificity, \'precision\': divide(\'precision\', tp, tp + fp),\n        \'F1\': divide(\'F1\', 2 * tp, 2 * tp + fp + fn), \'accuracy\': (tp + tn) / len(labels),\n        \'balanced_accuracy\': None if sensitivity is None or specificity is None else (sensitivity + specificity) / 2,\n        \'average_precision\': None, \'ROC_AUC\': None, \'threshold\': threshold, \'positive_class\': \'rd\'}\n    if set(labels.tolist()) == {0, 1}:\n        value[\'average_precision\'] = float(average_precision_score(labels, probabilities))\n        value[\'ROC_AUC\'] = float(roc_auc_score(labels, probabilities))\n    else:\n        reasons.update(average_precision=\'requires both classes for this protocol\', ROC_AUC=\'requires both classes\',\n                       balanced_accuracy=\'requires both classes\')\n    value[\'undefined_reasons\'] = reasons\n    return value\n\n\ndef probe_training_precision(model_factory, inputs, labels, physical_batch=4, precision=\'16-mixed\'):\n    """Disposable real-input timing; stabilize the scaler before measured updates."""\n    if not torch.cuda.is_available():\n        raise RuntimeError(\'The precision probe requires CUDA\')\n    model = optimizer = x = y = loss = scaler = None\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    try:\n        model = model_factory().cuda()\n        optimizer = torch.optim.Adam(model.parameters(), lr=1.5e-5)\n        scaler = torch.amp.GradScaler(\'cuda\', enabled=precision == \'16-mixed\')\n        indices = torch.arange(physical_batch) % len(inputs)\n        x, y = inputs[indices].cuda(), labels[indices].cuda()\n        timings, skipped_measured = [], 0\n        for iteration in range(13):\n            optimizer.zero_grad(set_to_none=True)\n            torch.cuda.synchronize()\n            start = time.perf_counter()\n            with autocast_for(\'cuda\', precision):\n                loss = nn.BCEWithLogitsLoss()(model(x).reshape(-1), y)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite precision-probe loss\')\n            scaler.scale(loss).backward()\n            scaler.unscale_(optimizer)\n            finite = all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.parameters())\n            nonzero = finite and any(torch.count_nonzero(p.grad).item() for p in model.parameters())\n            old_scale = scaler.get_scale()\n            scaler.step(optimizer)\n            scaler.update()\n            torch.cuda.synchronize()\n            if iteration >= 8:\n                skipped_measured += int(scaler.get_scale() < old_scale)\n                if not finite or not nonzero:\n                    raise ValueError(\'Precision probe did not stabilize with finite nonzero gradients\')\n                timings.append(time.perf_counter() - start)\n        if skipped_measured:\n            raise ValueError(\'Measured precision-probe updates were skipped\')\n        return {\'status\': \'passed\', \'precision\': precision, \'physical_batch\': physical_batch,\n            \'training_step_seconds_median\': float(np.median(timings)),\n            \'peak_allocated_bytes\': torch.cuda.max_memory_allocated(),\n            \'peak_reserved_bytes\': torch.cuda.max_memory_reserved(),\n            \'loss_last\': loss.item(), \'final_scale\': scaler.get_scale(),\n            \'measured_skipped_updates\': skipped_measured, \'finite_nonzero_gradients\': True,\n            \'weights_discarded\': True, \'test_used\': False}\n    finally:\n        del model, optimizer, x, y, loss, scaler\n        torch.cuda.empty_cache()\n\n\nclass CheckpointStore:\n    """Two alternating files plus an atomic pointer retain the last valid generation."""\n\n    def __init__(self, directory):\n        self.directory = Path(directory)\n        self.directory.mkdir(parents=True, exist_ok=True)\n        self.pointer = self.directory / \'last_checkpoint.json\'\n\n    def save(self, payload):\n        previous = json.loads(self.pointer.read_text()) if self.pointer.exists() else None\n        slot = \'last_b.pt\' if previous and previous[\'file\'] == \'last_a.pt\' else \'last_a.pt\'\n        target = self.directory / slot\n        temporary = target.with_suffix(\'.pt.tmp\')\n        torch.save(payload, temporary)\n        digest = file_digest(temporary)\n        temporary.replace(target)\n        write_json_atomic(self.pointer, {\'file\': slot, \'sha256\': digest,\n            \'previous\': None if previous is None else {\'file\': previous[\'file\'], \'sha256\': previous[\'sha256\']}})\n\n    def load(self):\n        if not self.pointer.exists():\n            return None\n        current = json.loads(self.pointer.read_text())\n        for candidate in [current, current.get(\'previous\')]:\n            if not candidate:\n                continue\n            path = self.directory / candidate[\'file\']\n            if candidate[\'file\'] not in {\'last_a.pt\', \'last_b.pt\'}:\n                raise ValueError(\'Unexpected checkpoint filename\')\n            if path.is_file() and file_digest(path) == candidate[\'sha256\']:\n                payload = torch.load(path, map_location=\'cpu\', weights_only=True)\n                if candidate is not current:\n                    # Repair the pointer before any save can overwrite the fallback generation.\n                    write_json_atomic(self.pointer, {**candidate, \'previous\': None})\n                return payload\n        raise ValueError(\'No valid checkpoint generation; files and hashes disagree\')\n\n\ndef run_training(model_factory, train_dataset, validation_dataset, config, directory,\n                 data_fingerprint, runtime, device=\'cuda\', max_epochs_this_call=1, report=print):\n    """Resume a fixed protocol, including interrupted epochs at update boundaries."""\n    directory = Path(directory)\n    directory.mkdir(parents=True, exist_ok=True)\n    signature = json_digest(config)\n    store = CheckpointStore(directory)\n    checkpoint = store.load()\n    if checkpoint is not None:\n        if checkpoint[\'schema_version\'] != 1 or checkpoint[\'config_digest\'] != signature:\n            raise ValueError(\'Checkpoint configuration differs; choose a new run directory\')\n        if checkpoint[\'data_fingerprint\'] != data_fingerprint:\n            raise ValueError(\'Checkpoint data/decoder/cache fingerprint differs; do not resume on changed inputs\')\n        if checkpoint[\'runtime\'] != runtime:\n            raise ValueError(\'Runtime versions/GPU differ from the checkpoint; inspect before changing the fixed run\')\n    elif (directory / \'history.json\').exists():\n        raise ValueError(\'History exists without a valid checkpoint; retain artifacts and inspect before restarting\')\n    device = torch.device(device)\n    precision = config[\'training\'][\'precision\']\n    if precision == \'16-mixed\' and device.type != \'cuda\':\n        raise ValueError(\'Mixed-precision training must use CUDA\')\n    seed_training(config[\'seed\'])\n    model = model_factory().to(device)\n    settings = config[\'training\']\n    optimizer = torch.optim.Adam(model.parameters(), lr=settings[\'learning_rate\'],\n        betas=tuple(settings[\'betas\']), eps=settings[\'eps\'], weight_decay=settings[\'weight_decay\'])\n    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode=\'min\',\n        factor=.1, patience=10, threshold=1e-4, threshold_mode=\'rel\', cooldown=0, min_lr=0., eps=1e-8)\n    scaler = torch.amp.GradScaler(\'cuda\', enabled=precision == \'16-mixed\')\n    history, best_accuracy, best_epoch, best_state = [], None, None, None\n    first_epoch, resume_progress = 1, {}\n    if checkpoint is not None:\n        model.load_state_dict(checkpoint[\'model_state\'])\n        optimizer.load_state_dict(checkpoint[\'optimizer_state\'])\n        scheduler.load_state_dict(checkpoint[\'scheduler_state\'])\n        scaler.load_state_dict(checkpoint[\'scaler_state\'])\n        history, best_accuracy, best_epoch = checkpoint[\'history\'], checkpoint[\'best_accuracy\'], checkpoint[\'best_epoch\']\n        best_state = checkpoint[\'best_model_state\']\n        first_epoch = checkpoint[\'epoch\'] + int(checkpoint[\'phase\'] == \'epoch_complete\')\n        if checkpoint[\'phase\'] == \'train\':\n            resume_progress = checkpoint[\'progress\']\n        restore_random_state(checkpoint[\'random_state\'])\n        report(f\'Resuming epoch {first_epoch}, training cursor {resume_progress.get("position", 0)}\')\n    # Rebuild human-readable history from the authoritative checkpoint after interrupted writes.\n    write_json_atomic(directory / \'history.json\', history)\n    write_json_atomic(directory / \'resolved_optimizer.json\', {\'optimizer\': optimizer.__class__.__name__,\n        \'defaults\': optimizer.defaults, \'scheduler\': {\'name\': \'ReduceLROnPlateau\',\n        **{key: value for key, value in scheduler.state_dict().items()\n           if key in {\'mode\', \'factor\', \'patience\', \'threshold\', \'threshold_mode\', \'cooldown\', \'min_lrs\', \'eps\'}}},\n        \'precision\': precision})\n    target_epochs = settings[\'epochs\']\n    stop_epoch = target_epochs if max_epochs_this_call is None else min(target_epochs, first_epoch + max_epochs_this_call - 1)\n    if max_epochs_this_call is not None and max_epochs_this_call < 1:\n        raise ValueError(\'Session epoch limit must be positive or None\')\n    training_start = time.perf_counter()\n    for epoch in range(first_epoch, stop_epoch + 1):\n        epoch_start = time.perf_counter()\n        resumed_from = resume_progress.get(\'position\', 0)\n        order = epoch_order(len(train_dataset), config[\'seed\'], epoch)\n        if checkpoint is not None and resume_progress and checkpoint[\'order\'] != order:\n            raise ValueError(\'Saved epoch order differs from the fixed seed/order protocol\')\n        def make_payload(phase, progress):\n            return {\'schema_version\': 1, \'config_digest\': signature, \'config\': config,\n                \'data_fingerprint\': data_fingerprint, \'runtime\': runtime,\n                \'epoch\': epoch, \'phase\': phase, \'order\': order, \'progress\': progress,\n                \'model_state\': model.state_dict(), \'optimizer_state\': optimizer.state_dict(),\n                \'scheduler_state\': scheduler.state_dict(), \'scaler_state\': scaler.state_dict(),\n                \'history\': history, \'best_accuracy\': best_accuracy, \'best_epoch\': best_epoch,\n                \'best_model_state\': best_state, \'random_state\': capture_random_state(), \'test_used\': False}\n        # A checkpoint exists before the first batch, including the initialized model.\n        if not resume_progress:\n            resume_progress = {\'position\': 0, \'loss_sum\': 0., \'examples_seen\': 0,\n                               \'optimizer_updates\': 0, \'skipped_updates\': 0}\n            store.save(make_payload(\'train\', resume_progress))\n        def boundary(progress):\n            if progress[\'optimizer_updates\'] % settings[\'checkpoint_every_updates\'] == 0 or progress[\'position\'] == len(train_dataset):\n                store.save(make_payload(\'train\', progress))\n                write_json_atomic(directory / \'run_status.json\', {\'status\': \'training\',\n                    \'epoch\': epoch, \'training_examples_seen\': progress[\'position\'],\n                    \'training_examples_total\': len(train_dataset), \'completed_epochs\': len(history), \'test_used\': False})\n                report(f\'Epoch {epoch}/{target_epochs}: {progress["position"]}/{len(train_dataset)} training clips; checkpoint saved\')\n            elif progress[\'optimizer_updates\'] % 2 == 0:\n                report(f\'Epoch {epoch}/{target_epochs}: {progress["position"]}/{len(train_dataset)} training clips\')\n        progress = train_epoch_accumulated(model, train_dataset, order, optimizer, device,\n            settings[\'physical_batch\'], settings[\'effective_batch\'], precision, scaler,\n            on_boundary=boundary, **{key: value for key, value in resume_progress.items() if key != \'position\'},\n            start_position=resume_progress[\'position\'])\n        validation_start = time.perf_counter()\n        validation_loss, labels, probabilities = validate_model(model, validation_dataset, device,\n            settings[\'physical_batch\'], precision)\n        metrics = binary_metrics(labels, probabilities, config[\'evaluation\'][\'threshold\'])\n        row = {\'epoch\': epoch, \'train_loss\': progress[\'train_loss\'], \'validation_loss\': validation_loss,\n            \'resumed_from_position\': resumed_from,\n            \'learning_rate_used\': optimizer.param_groups[0][\'lr\'], \'seconds_this_segment\': time.perf_counter() - epoch_start,\n            \'validation_seconds\': time.perf_counter() - validation_start,\n            \'skipped_updates\': progress[\'skipped_updates\'], **metrics}\n        scheduler.step(validation_loss)\n        history.append(row)\n        is_best = best_accuracy is None or metrics[\'accuracy\'] > best_accuracy\n        if is_best:\n            best_accuracy, best_epoch = metrics[\'accuracy\'], epoch\n            best_state = {key: value.detach().cpu().clone() for key, value in model.state_dict().items()}\n        prediction = {\'dataset_labels\': labels.tolist(), \'rd_probabilities\': probabilities.tolist()}\n        write_json_atomic(directory / f\'validation_epoch_{epoch:03d}.json\', prediction)\n        store.save(make_payload(\'epoch_complete\', progress))\n        write_json_atomic(directory / \'history.json\', history)\n        if is_best:\n            write_json_atomic(directory / \'best_validation_predictions.json\', prediction)\n        report({key: row[key] for key in [\'epoch\', \'train_loss\', \'validation_loss\', \'accuracy\',\n                                        \'sensitivity\', \'specificity\', \'average_precision\', \'seconds_this_segment\']})\n        resume_progress = {}\n    # best.pt is a derived export; authoritative best weights also live in both latest checkpoint generations.\n    if best_state is not None:\n        temporary = directory / \'best.pt.tmp\'\n        torch.save({\'model_state\': best_state, \'epoch\': best_epoch, \'config\': config,\n                    \'data_fingerprint\': data_fingerprint, \'test_used\': False}, temporary)\n        temporary.replace(directory / \'best.pt\')\n        best_prediction = directory / f\'validation_epoch_{best_epoch:03d}.json\'\n        write_json_atomic(directory / \'best_validation_predictions.json\', json.loads(best_prediction.read_text()))\n    completed = len(history)\n    summary = {\'experiment_id\': config[\'experiment_id\'], \'epochs_completed\': completed,\n        \'target_epochs\': target_epochs, \'status\': \'completed\' if completed == target_epochs else \'partial\',\n        \'best_epoch\': best_epoch, \'best_validation_accuracy\': best_accuracy,\n        \'best_validation_metrics\': next((row for row in history if row[\'epoch\'] == best_epoch), None),\n        \'config_digest\': signature, \'data_fingerprint\': data_fingerprint, \'runtime\': runtime,\n        \'precision\': precision, \'training_clips\': len(train_dataset), \'validation_clips\': len(validation_dataset),\n        \'last_call_seconds\': time.perf_counter() - training_start, \'test_used\': False}\n    write_json_atomic(directory / \'summary.json\', summary)\n    write_json_atomic(directory / \'run_status.json\', summary)\n    return summary\n'
r2p1_load_module('unetpp_training', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""Verified full-video cache for the fixed UNet++ train/validation protocol."""\nimport json\nfrom pathlib import Path\nimport time\nfrom urllib.parse import quote\n\nimport numpy as np\nimport torch\nfrom torch.utils.data import Dataset\n\nfrom rd_ultrasound_ml.unetpp_baseline import prepare_unetpp_clip\nfrom rd_ultrasound_ml.unetpp_training import file_digest, json_digest, write_json_atomic\n\n\ndef verify_partitions(train, validation, metadata):\n    """Preserve supplied CSV order and reject dropped rows, overlap or label disagreement."""\n    merged = []\n    for frame, count, labels in [(train, 3873, {0: 3511, 1: 362}),\n                                  (validation, 431, {0: 391, 1: 40})]:\n        if len(frame) != count or not frame[\'path\'].is_unique or frame[\'label\'].value_counts().to_dict() != labels:\n            raise ValueError(\'Official partition size, uniqueness or label counts differ\')\n        joined = frame.merge(metadata, left_on=\'path\', right_on=\'file_path\', how=\'left\',\n                             sort=False, validate=\'one_to_one\')\n        if joined[\'clip_id\'].isna().any() or not joined[\'label\'].map({0: \'non_rd\', 1: \'rd\'}).eq(joined[\'diagnostic_class\']).all():\n            raise ValueError(\'Partition/metadata labels or paths disagree\')\n        if not joined[\'clip_id\'].is_unique or not joined[\'path\'].tolist() == frame[\'path\'].tolist():\n            raise ValueError(\'Manifest identity/order changed\')\n        for path in joined[\'file_path\']:\n            if Path(path).is_absolute() or \'..\' in Path(path).parts:\n                raise ValueError(\'Unsafe relative clip path\')\n        merged.append(joined)\n    if set(train[\'path\']) & set(validation[\'path\']):\n        raise ValueError(\'Training and validation clips overlap\')\n    return tuple(merged)\n\n\ndef record_video_sources(frame, revision, path, api, report=print):\n    """Persist each verified API chunk, using immutable-revision LFS hashes."""\n    path = Path(path)\n    catalog = json.loads(path.read_text()) if path.exists() else {\n        \'dataset\': \'pcvlab/erdes\', \'revision\': revision,\n        \'license_declaration\': \'cc-by-4.0 (author dataset card)\', \'files\': {}}\n    if catalog[\'revision\'] != revision or catalog[\'dataset\'] != \'pcvlab/erdes\':\n        raise ValueError(\'Source catalog revision differs\')\n    files = catalog[\'files\']\n    missing = [value for value in frame[\'file_path\'] if value not in files]\n    for start in range(0, len(missing), 50):\n        requested = [\'clips/\' + value for value in missing[start:start + 50]]\n        for attempt in range(5):\n            try:\n                entries = api.get_paths_info(\'pcvlab/erdes\', paths=requested, repo_type=\'dataset\',\n                                             revision=revision, token=False)\n                break\n            except Exception as error:\n                response = getattr(error, \'response\', None)\n                status = getattr(response, \'status_code\', None)\n                if status not in {408, 429, 500, 502, 503, 504} or attempt == 4:\n                    raise\n                retry_after = response.headers.get(\'Retry-After\', \'\')\n                delay = max(2 ** (attempt + 1), float(retry_after) if retry_after.isdigit() else 0)\n                if delay > 60:\n                    raise RuntimeError(f\'Source API requests a wait of {delay}s; wait, then rerun preparation\') from None\n                report(f\'Source catalog HTTP {status}; retry in {delay}s\')\n                time.sleep(delay)\n        by_path = {entry.path: entry for entry in entries}\n        if set(by_path) != set(requested):\n            raise ValueError(\'Source API omitted selected clips\')\n        for requested_path in requested:\n            entry = by_path[requested_path]\n            if entry.lfs is None or len(entry.lfs.sha256) != 64 or entry.size != entry.lfs.size:\n                raise ValueError(f\'Missing source SHA256/size: {requested_path}\')\n            files[requested_path.removeprefix(\'clips/\')] = {\n                \'sha256\': entry.lfs.sha256, \'bytes\': int(entry.size),\n                \'url\': f\'https://huggingface.co/datasets/pcvlab/erdes/resolve/{revision}/{quote(requested_path, safe="/")}\'}\n        write_json_atomic(path, catalog)\n        report(f\'Source hashes recorded: {len(files)} clips\')\n    for value in frame[\'file_path\']:\n        source = files[value]\n        expected_url = f\'https://huggingface.co/datasets/pcvlab/erdes/resolve/{revision}/clips/{quote(value, safe="/")}\'\n        if (len(source[\'sha256\']) != 64 or any(c not in \'0123456789abcdef\' for c in source[\'sha256\'])\n                or source[\'bytes\'] < 1 or source[\'url\'] != expected_url):\n            raise ValueError(f\'Invalid saved source provenance: {value}\')\n    return files\n\n\ndef prepare_cached_clip(row, source, data_dir, cache_dir, cache_signature, download, target=(96, 128, 128)):\n    """Commit a tensor only after decode/range checks; validate hashes on every reuse."""\n    data_dir, cache_dir = Path(data_dir), Path(cache_dir)\n    cache_dir.mkdir(parents=True, exist_ok=True)\n    identity = {\'signature\': cache_signature, \'clip_id\': row.clip_id, \'file_path\': row.file_path,\n        \'source_sha256\': source[\'sha256\'], \'frame_count\': int(row.frame_count),\n        \'height\': int(row.height), \'width\': int(row.width), \'target\': list(target)}\n    name = json_digest(identity)\n    tensor_path, check_path = cache_dir / (name + \'.npy\'), cache_dir / (name + \'.json\')\n    if check_path.exists():\n        check = json.loads(check_path.read_text())\n        if check[\'identity\'] != identity or not tensor_path.is_file() or file_digest(tensor_path) != check[\'tensor_sha256\']:\n            raise ValueError(f\'Cached tensor integrity failure: {row.clip_id}; retain the file for inspection\')\n        tensor = np.load(tensor_path, mmap_mode=\'r\', allow_pickle=False)\n        if tensor.shape != (1, *target) or tensor.dtype != np.float32:\n            raise ValueError(f\'Cached tensor shape/dtype failure: {row.clip_id}\')\n        return tensor_path, check\n    video_path = data_dir / \'clips\' / row.file_path\n    download(source[\'url\'], video_path, source[\'sha256\'], expected_bytes=source[\'bytes\'])\n    tensor = prepare_unetpp_clip(video_path, int(row.frame_count), int(row.height), int(row.width), target=target)\n    temporary = tensor_path.with_suffix(\'.npy.tmp\')\n    with temporary.open(\'wb\') as stream:\n        np.save(stream, tensor.numpy(), allow_pickle=False)\n    check = {\'identity\': identity, \'shape\': list(tensor.shape), \'dtype\': \'float32\',\n        \'min\': float(tensor.min()), \'max\': float(tensor.max()), \'tensor_sha256\': file_digest(temporary)}\n    temporary.replace(tensor_path)\n    write_json_atomic(check_path, check)\n    return tensor_path, check\n\n\nclass CachedClips(Dataset):\n    def __init__(self, paths, labels):\n        self.paths, self.labels = list(map(Path, paths)), np.asarray(labels, dtype=np.float32)\n        if len(self.paths) == 0 or len(self.paths) != len(self.labels) or not np.isin(self.labels, [0, 1]).all():\n            raise ValueError(\'Expected matching non-empty cached paths and binary labels\')\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, index):\n        # One file at a time; no 27 GB concatenation or retained memmap handles.\n        array = np.load(self.paths[index], allow_pickle=False)\n        return torch.from_numpy(array), torch.tensor(self.labels[index])\n'
r2p1_load_module('unetpp_data', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""Kaggle CPU-output shards and verified input/checkpoint handoff.\n\nPlatform/storage adaptation only: arrays retain the original float32 values.\n"""\nimport io\nimport json\nfrom pathlib import Path\nimport shutil\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\n\nfrom rd_ultrasound_ml.unetpp_data import verify_partitions\nfrom rd_ultrasound_ml.unetpp_training import CheckpointStore, file_digest, json_digest\n\nMETADATA_MD5 = \'95828fb5bdffdbf8173b5bcfc99e8148\'\nSPLITS_MD5 = \'c5630e3f056ce358dcd384aaed946e64\'\n\n\ndef shard_indices(length, index, count=2):\n    if length < count or count < 1 or not 0 <= index < count:\n        raise ValueError(\'Invalid preparation shard dimensions\')\n    return np.array_split(np.arange(length), count)[index].tolist()\n\n\ndef read_official_inputs(directory):\n    import hashlib\n    directory = Path(directory)\n    metadata = (directory / \'erdes_metadata.csv\').read_bytes()\n    splits = (directory / \'splits.zip\').read_bytes()\n    if hashlib.md5(metadata).hexdigest() != METADATA_MD5 or hashlib.md5(splits).hexdigest() != SPLITS_MD5:\n        raise ValueError(\'Official metadata/split archive checksum differs\')\n    with zipfile.ZipFile(io.BytesIO(splits)) as archive:\n        train = pd.read_csv(io.BytesIO(archive.read(\'splits/non_rd_vs_rd/train.csv\')))\n        validation = pd.read_csv(io.BytesIO(archive.read(\'splits/non_rd_vs_rd/val.csv\')))\n    return verify_partitions(train, validation, pd.read_csv(io.BytesIO(metadata)))\n\n\ndef load_prepared_shards(input_root, config, report=print):\n    """Reject incomplete/duplicate shards and verify every attached tensor before use."""\n    input_root = Path(input_root)\n    manifests = sorted(input_root.rglob(\'prepared_manifest.json\'))\n    if len(manifests) != 2:\n        raise ValueError(f\'Attach exactly two completed CPU preparation outputs; found {len(manifests)} manifests\')\n    train, validation = read_official_inputs(manifests[0].parent)\n    all_rows = pd.concat([train, validation], ignore_index=True)\n    records, paths, seen_shards = {}, {}, set()\n    identity = None\n    for path in manifests:\n        manifest = json.loads(path.read_text())\n        if manifest.get(\'complete\') is not True or manifest.get(\'shard_count\') != 2:\n            raise ValueError(\'Preparation output is incomplete or has the wrong shard count\')\n        index = manifest[\'shard_index\']\n        if index in seen_shards or index not in {0, 1}:\n            raise ValueError(\'Duplicate or invalid preparation shard\')\n        seen_shards.add(index)\n        if manifest[\'config_digest\'] != json_digest(config):\n            raise ValueError(\'CPU preparation configuration/code differs from this training notebook\')\n        if identity is None:\n            identity = manifest[\'preprocessing_identity\']\n        elif identity != manifest[\'preprocessing_identity\']:\n            raise ValueError(\'CPU shard preprocessing versions differ\')\n        expected = shard_indices(len(all_rows), index)\n        entries = manifest[\'entries\']\n        if [entry[\'global_index\'] for entry in entries] != expected:\n            raise ValueError(\'Preparation shard omitted, duplicated or reordered selected clips\')\n        other_train, other_val = read_official_inputs(path.parent)\n        if not train.equals(other_train) or not validation.equals(other_val):\n            raise ValueError(\'CPU shards contain different official partitions\')\n        for entry in entries:\n            position = entry[\'global_index\']\n            row = all_rows.iloc[position]\n            check = entry[\'check\']\n            if entry[\'clip_id\'] != row[\'clip_id\'] or entry[\'label\'] != int(row[\'label\']):\n                raise ValueError(\'Prepared clip identity/label differs from official manifest\')\n            expected_identity = {\'signature\': json_digest(identity), \'clip_id\': row[\'clip_id\'],\n                \'file_path\': row[\'file_path\'], \'source_sha256\': entry[\'source\'][\'sha256\'],\n                \'frame_count\': int(row[\'frame_count\']), \'height\': int(row[\'height\']),\n                \'width\': int(row[\'width\']), \'target\': [96, 128, 128]}\n            if check[\'identity\'] != expected_identity:\n                raise ValueError(\'Prepared clip provenance differs from selected source/preprocessing\')\n            relative = Path(entry[\'array\'])\n            if relative.is_absolute() or \'..\' in relative.parts:\n                raise ValueError(\'Unsafe prepared-array path\')\n            array_path = path.parent / relative\n            if file_digest(array_path) != check[\'tensor_sha256\']:\n                raise ValueError(f\'Prepared tensor checksum differs: {row["clip_id"]}\')\n            array = np.load(array_path, mmap_mode=\'r\', allow_pickle=False)\n            if array.dtype != np.float32 or array.shape != (1, 96, 128, 128):\n                raise ValueError(\'Prepared tensor shape/dtype differs from fixed protocol\')\n            paths[position], records[position] = array_path, entry\n            if len(paths) % 100 == 0:\n                report(f\'Verified attached input {len(paths)}/{len(all_rows)}\')\n    if set(paths) != set(range(len(all_rows))):\n        raise ValueError(\'Prepared inputs do not cover the complete official train/validation sets\')\n    ordered_records = [records[index] for index in range(len(all_rows))]\n    fingerprint = json_digest({\'config_digest\': json_digest(config), \'preprocessing_identity\': identity,\n        \'records\': ordered_records,\n        \'train_manifest\': train.to_csv(index=False), \'validation_manifest\': validation.to_csv(index=False)})\n    return train, validation, [paths[index] for index in range(len(all_rows))], fingerprint, identity\n\n\ndef import_previous_run(input_root, destination, config, previous_required=False):\n    """Copy only saved training artifacts; never write to attached read-only inputs."""\n    destination = Path(destination)\n    candidates = sorted(Path(input_root).rglob(\'last_checkpoint.json\'))\n    if (destination / \'last_checkpoint.json\').exists():\n        return destination  # Current interactive session already owns the latest state.\n    if len(candidates) > 1:\n        raise ValueError(\'Attach only one preceding training output to avoid ambiguous resume\')\n    if not candidates:\n        if previous_required:\n            raise ValueError(\'This continuation requires the preceding successful GPU notebook output\')\n        return None\n    source = candidates[0].parent\n    previous_config = json.loads((source / \'config.json\').read_text())\n    if previous_config != config:\n        raise ValueError(\'Preceding training configuration differs; do not initialize over it\')\n    destination.mkdir(parents=True, exist_ok=True)\n    # Include the per-epoch prediction records needed to recover the selected best epoch.\n    for path in source.iterdir():\n        if path.is_file() and path.suffix in {\'.json\', \'.csv\', \'.pt\', \'.py\'}:\n            shutil.copy2(path, destination / path.name)\n    if CheckpointStore(destination).load() is None:\n        raise ValueError(\'Preceding training output lacks a valid checkpoint\')\n    return source\n'
r2p1_load_module('unetpp_kaggle', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""Reuse the verified U-Net++ preparation cache under a new experiment identity.\n\n``load_prepared_shards`` compares each preparation manifest\'s ``config_digest``\nwith the digest of the config it receives. That digest covers the original\npreparation config (including its model/code identity), so passing a new\nmodel\'s training config would reject a valid cache. This adapter does not\nrewrite manifests or weaken those checks. It reads the preparation config that\neach CPU job saved beside its manifest, verifies it against that manifest and\nagainst the digest pinned in the new training config, checks that the two\nshare preprocessing/dataset identity, then calls the unchanged\n``load_prepared_shards`` with the original preparation config. All tensor/source\nhashes, label/partition/order and shape checks still run. The returned run\nfingerprint binds the preparation identity, the cache fingerprint and the new\ntraining config digest, so checkpoints cannot resume across any of them.\n"""\nimport json\nfrom pathlib import Path\n\nfrom rd_ultrasound_ml.unetpp_kaggle import load_prepared_shards\nfrom rd_ultrasound_ml.unetpp_training import json_digest\n\nSHARED_FIELDS = (\'dataset_revision\', \'task\', \'label_mapping\', \'preprocessing\')\n\n\ndef read_preparation_config(input_root):\n    """Return the one preparation config shared by both attached shards and its digest."""\n    manifests = sorted(Path(input_root).rglob(\'prepared_manifest.json\'))\n    if len(manifests) != 2:\n        raise ValueError(f\'Attach exactly two completed CPU preparation outputs; found {len(manifests)} manifests\')\n    configs = []\n    for path in manifests:\n        config_path = path.parent / \'config.json\'\n        if not config_path.is_file():\n            raise ValueError(f\'Preparation output lacks its saved config.json: {path.parent}\')\n        config = json.loads(config_path.read_text())\n        if json_digest(config) != json.loads(path.read_text()).get(\'config_digest\'):\n            raise ValueError(\'Saved preparation config does not match its manifest digest\')\n        configs.append(config)\n    if configs[0] != configs[1]:\n        raise ValueError(\'Attached preparation shards were produced by different configs\')\n    return configs[0], json_digest(configs[0])\n\n\ndef load_cache_for_run(input_root, training_config, report=print):\n    """Validate the attached cache for a new training identity; never modifies inputs."""\n    cache = training_config.get(\'cache\')\n    if not isinstance(cache, dict) or len(str(cache.get(\'preparation_config_digest\', \'\'))) != 64:\n        raise ValueError(\'Training config must pin cache.preparation_config_digest\')\n    preparation, preparation_digest = read_preparation_config(input_root)\n    if preparation_digest != cache[\'preparation_config_digest\']:\n        raise ValueError(\'Attached preparation config differs from the pinned preparation identity\')\n    if preparation.get(\'experiment_id\') != cache.get(\'preparation_experiment_id\'):\n        raise ValueError(\'Attached preparation experiment differs from the pinned one\')\n    for field in SHARED_FIELDS:\n        if preparation.get(field) != training_config.get(field):\n            raise ValueError(f\'Training config {field} differs from the cached preparation\')\n    train, validation, paths, cache_fingerprint, identity = load_prepared_shards(\n        input_root, preparation, report=report)\n    expected = cache.get(\'expected_cache_fingerprint\')\n    if expected is not None and cache_fingerprint != expected:\n        raise ValueError(\'Verified cache fingerprint differs from the pinned reference fingerprint\')\n    record = {\'preparation_experiment_id\': preparation[\'experiment_id\'],\n        \'preparation_config_digest\': preparation_digest, \'cache_fingerprint\': cache_fingerprint,\n        \'preprocessing_identity\': identity, \'training_config_digest\': json_digest(training_config)}\n    run_fingerprint = json_digest(record)\n    return train, validation, paths, run_fingerprint, {**record, \'run_fingerprint\': run_fingerprint}\n'
r2p1_load_module('video_cache', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""Model-agnostic probe/export helpers for the final-milestone comparison runs.\n\nValidation-only: nothing here opens test data. Reuses the U-Net++ training\nengine\'s metric, validation and precision-probe functions unchanged.\n"""\nimport json\nfrom pathlib import Path\n\nimport numpy as np\nimport torch\n\nfrom rd_ultrasound_ml.unetpp_training import (binary_metrics, file_digest, probe_training_precision,\n    validate_model)\n\n\ndef probe_physical_batches(model_factory, inputs, labels, batches=(4, 2, 1), precision=\'16-mixed\'):\n    """Disposable probes in order; stops after the first batch that fits. Weights are discarded."""\n    results = []\n    for batch in batches:\n        try:\n            result = probe_training_precision(model_factory, inputs, labels, batch, precision)\n        except torch.cuda.OutOfMemoryError:\n            torch.cuda.empty_cache()\n            result = {\'status\': \'out_of_memory\', \'precision\': precision, \'physical_batch\': batch,\n                \'weights_discarded\': True, \'test_used\': False}\n        results.append(result)\n        if result[\'status\'] == \'passed\':\n            break\n    return results\n\n\ndef prediction_rows(manifest, labels, probabilities, threshold, partition):\n    """One recomputable row per clip, in manifest order; rejects any label/order disagreement."""\n    labels, probabilities = np.asarray(labels), np.asarray(probabilities, dtype=np.float64)\n    if len(manifest) != len(labels) or len(labels) != len(probabilities):\n        raise ValueError(\'Prediction count differs from the manifest\')\n    if not np.array_equal(manifest[\'label\'].to_numpy().astype(int), labels.astype(int)):\n        raise ValueError(\'Prediction labels/order differ from the manifest\')\n    return [{\'clip_id\': clip_id, \'dataset_label\': int(label), \'rd_probability\': float(probability),\n             \'predicted_class\': int(probability >= threshold), \'threshold\': threshold, \'partition\': partition}\n            for clip_id, label, probability in zip(manifest[\'clip_id\'], labels, probabilities)]\n\n\ndef verify_best_checkpoint(model_factory, directory, validation_dataset, device, physical_batch,\n                           precision, threshold=.5, tolerance=1e-4):\n    """Reload best.pt into a fresh model and reproduce the saved selected-epoch validation probabilities."""\n    directory = Path(directory)\n    summary = json.loads((directory / \'summary.json\').read_text())\n    saved = json.loads((directory / \'best_validation_predictions.json\').read_text())\n    payload = torch.load(directory / \'best.pt\', map_location=\'cpu\', weights_only=True)\n    if payload[\'epoch\'] != summary[\'best_epoch\']:\n        raise ValueError(\'best.pt epoch differs from the summary-selected epoch\')\n    model = model_factory()\n    model.load_state_dict(payload[\'model_state\'])\n    model.to(device)\n    _, labels, probabilities = validate_model(model, validation_dataset, device, physical_batch, precision)\n    if labels.tolist() != saved[\'dataset_labels\']:\n        raise ValueError(\'Reloaded validation labels/order differ from saved predictions\')\n    difference = float(np.max(np.abs(probabilities - np.asarray(saved[\'rd_probabilities\']))))\n    saved_metrics = binary_metrics(saved[\'dataset_labels\'], saved[\'rd_probabilities\'], threshold)\n    reloaded_metrics = binary_metrics(labels, probabilities, threshold)\n    same_counts = all(saved_metrics[key] == reloaded_metrics[key] for key in [\'TP\', \'TN\', \'FP\', \'FN\'])\n    result = {\'checkpoint\': \'best.pt\', \'checkpoint_sha256\': file_digest(directory / \'best.pt\'),\n        \'selected_epoch\': payload[\'epoch\'], \'max_abs_probability_difference\': difference,\n        \'tolerance\': tolerance, \'confusion_counts_reproduced\': same_counts,\n        \'passed\': difference <= tolerance and same_counts, \'precision\': precision,\n        \'physical_batch\': physical_batch, \'partition\': \'validation\', \'test_used\': False}\n    if not result[\'passed\']:\n        raise ValueError(f\'Selected checkpoint reload did not reproduce validation predictions: {result}\')\n    return result\n'
r2p1_load_module('comparison_export', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""One-channel torchvision R(2+1)D-18 RD clip classifier (comparison model).\n\nArchitecture source: torchvision.models.video.r2plus1d_18, constructed with\nweights=None and num_classes=1 (Tran et al., CVPR 2018, arXiv:1711.11248).\nOnly the first stem convolution changes, from three input channels to one, so\nthe model accepts the existing grayscale cache. The stem\'s later temporal\nconvolution, all BatchNorm/ReLU layers, factorized residual blocks, pooling and\nLinear(512, 1) head are the builder\'s defaults. No pretrained weights are used.\n"""\nimport torch\nfrom torch import nn\nimport torchvision\nfrom torchvision.models.video import r2plus1d_18\n\nGRADCAM_LAYER = \'backbone.layer4\'\nSTEM_REPLACEMENT = {\'in_channels\': 1, \'out_channels\': 45, \'kernel_size\': (1, 7, 7),\n    \'stride\': (1, 2, 2), \'padding\': (0, 3, 3), \'bias\': False}\nINITIALIZATION = ("builder defaults for all modules; replacement stem[0] uses "\n    "kaiming_normal_(mode=\'fan_out\', nonlinearity=\'relu\'), the builder\'s Conv3d rule. "\n    "fan_out (45*1*7*7) does not depend on input channels, so the replacement "\n    "uses the same initialization distribution as the stock first convolution")\n\n\nclass R2Plus1D18RDClassifier(nn.Module):\n    """Wraps torchvision VideoResNet as ``backbone``; forward returns raw logits of shape [B]."""\n\n    def __init__(self):\n        super().__init__()\n        backbone = r2plus1d_18(weights=None, num_classes=1)\n        original = backbone.stem[0]\n        expected = {\'in_channels\': 3, \'out_channels\': 45, \'kernel_size\': (1, 7, 7),\n            \'stride\': (1, 2, 2), \'padding\': (0, 3, 3), \'bias\': False}\n        found = {\'in_channels\': original.in_channels, \'out_channels\': original.out_channels,\n            \'kernel_size\': original.kernel_size, \'stride\': original.stride,\n            \'padding\': original.padding, \'bias\': original.bias is not None}\n        if not isinstance(original, nn.Conv3d) or found != expected:\n            raise RuntimeError(f\'Installed torchvision {torchvision.__version__} stem differs: {found}\')\n        replacement = nn.Conv3d(1, original.out_channels, kernel_size=original.kernel_size,\n            stride=original.stride, padding=original.padding, dilation=original.dilation,\n            groups=original.groups, bias=False, padding_mode=original.padding_mode)\n        nn.init.kaiming_normal_(replacement.weight, mode=\'fan_out\', nonlinearity=\'relu\')\n        backbone.stem[0] = replacement\n        self.backbone = backbone\n\n    def forward(self, inputs):\n        if inputs.ndim != 5 or inputs.shape[1] != 1:\n            raise ValueError(f\'Expected B,1,D,H,W grayscale clips; received {tuple(inputs.shape)}\')\n        return self.backbone(inputs).squeeze(1)\n\n\ndef gradcam_layer(model):\n    """Return the final pre-pooling feature module (``backbone.layer4``)."""\n    return model.get_submodule(GRADCAM_LAYER)\n\n\ndef describe_model(model=None):\n    """Plain-data architecture record for configs/reports; builds a model when omitted."""\n    model = R2Plus1D18RDClassifier() if model is None else model\n    stem = model.backbone.stem\n    return {\'name\': \'torchvision_r2plus1d_18_grayscale_binary\',\n        \'builder\': \'torchvision.models.video.r2plus1d_18(weights=None, num_classes=1)\',\n        \'torchvision\': torchvision.__version__, \'torch\': str(torch.__version__),\n        \'pretrained\': False, \'stem_replacement\': {**STEM_REPLACEMENT,\n            \'kernel_size\': list(STEM_REPLACEMENT[\'kernel_size\']), \'stride\': list(STEM_REPLACEMENT[\'stride\']),\n            \'padding\': list(STEM_REPLACEMENT[\'padding\'])},\n        \'stem\': [repr(module) for module in stem],\n        \'initialization\': INITIALIZATION,\n        \'normalization\': \'BatchNorm3d builder defaults (eps=1e-5, momentum=0.1, track_running_stats=True)\',\n        \'head\': \'AdaptiveAvgPool3d(1), flatten, Linear(512,1); squeeze to [B] raw logits\',\n        \'gradcam_layer\': GRADCAM_LAYER,\n        \'parameter_count\': sum(parameter.numel() for parameter in model.parameters())}\n'
r2p1_load_module('r2plus1d18_baseline', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""Frozen-checkpoint official-test evaluation; contains no training or tuning."""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom rd_ultrasound_ml.unetpp_data import CachedClips\nfrom rd_ultrasound_ml.unetpp_kaggle import read_official_inputs\nfrom rd_ultrasound_ml.unetpp_training import (\n    autocast_for, binary_metrics, file_digest, json_digest, write_json_atomic,\n)\n\n\ndef verify_frozen_training(directory, freeze):\n    """Check the selected artifact before reading any test partition."""\n    directory = Path(directory)\n    if (freeze[\'training_epochs_completed\'] != 50 or freeze[\'threshold\'] != .5\n            or freeze[\'test_accessed_at_freeze\'] is not False or freeze[\'checkpoint_file\'] != \'best.pt\'):\n        raise ValueError(\'Invalid pre-test freeze: require complete training and the fixed protocol\')\n    config = json.loads((directory/\'config.json\').read_text())\n    summary = json.loads((directory/\'summary.json\').read_text())\n    if (json_digest(config) != freeze[\'training_config_digest\']\n            or summary[\'config_digest\'] != freeze[\'training_config_digest\']\n            or summary[\'data_fingerprint\'] != freeze[\'training_data_fingerprint\']):\n        raise ValueError(\'Frozen training configuration/data fingerprint differs\')\n    if (summary[\'status\'] != \'completed\' or summary[\'epochs_completed\'] != 50\n            or summary[\'best_epoch\'] != freeze[\'selected_epoch\'] or summary[\'test_used\']):\n        raise ValueError(\'Training is incomplete or selected epoch differs from freeze\')\n    if file_digest(directory/\'best.pt\') != freeze[\'checkpoint_sha256\']:\n        raise ValueError(\'Selected checkpoint SHA256 differs from pre-test freeze\')\n    return config\n\n\ndef find_frozen_training(input_root, freeze):\n    matches = [path.parent for path in Path(input_root).rglob(\'best.pt\')\n               if file_digest(path) == freeze[\'checkpoint_sha256\']]\n    if len(matches) != 1:\n        raise ValueError(\'Attach exactly one matching final-training output for the frozen checkpoint\')\n    verify_frozen_training(matches[0], freeze)\n    return matches[0]\n\n\ndef validate_test_partition(test, metadata, train, validation):\n    if (len(test) != 1077 or not test[\'path\'].is_unique\n            or test[\'label\'].value_counts().to_dict() != {0: 977, 1: 100}):\n        raise ValueError(\'Official test count, uniqueness or labels differ\')\n    if set(test[\'path\']) & (set(train[\'path\']) | set(validation[\'path\'])):\n        raise ValueError(\'Test clips overlap training or validation\')\n    joined = test.merge(metadata, left_on=\'path\', right_on=\'file_path\', how=\'left\',\n                        sort=False, validate=\'one_to_one\')\n    if (joined[\'clip_id\'].isna().any() or not joined[\'clip_id\'].is_unique\n            or not joined[\'label\'].map({0:\'non_rd\', 1:\'rd\'}).eq(joined[\'diagnostic_class\']).all()\n            or joined[\'path\'].tolist() != test[\'path\'].tolist()):\n        raise ValueError(\'Test manifest/metadata identity, labels or order differ\')\n    for value in joined[\'file_path\']:\n        if Path(value).is_absolute() or \'..\' in Path(value).parts:\n            raise ValueError(\'Unsafe test clip path\')\n    return joined\n\n\ndef read_frozen_test_inputs(source_directory, training_directory, freeze):\n    verify_frozen_training(training_directory, freeze)  # Must precede even opening test.csv.\n    source_directory = Path(source_directory)\n    train, validation = read_official_inputs(source_directory)  # Checks official archive/metadata MD5s.\n    with zipfile.ZipFile(source_directory/\'splits.zip\') as archive:\n        content = archive.read(\'splits/non_rd_vs_rd/test.csv\')\n    metadata = pd.read_csv(source_directory/\'erdes_metadata.csv\')\n    frame = validate_test_partition(pd.read_csv(io.BytesIO(content)), metadata, train, validation)\n    return frame, hashlib.sha256(content).hexdigest()\n\n\ndef load_test_cache(input_root, training_directory, freeze, report=print):\n    manifests = list(Path(input_root).rglob(\'test_prepared_manifest.json\'))\n    if len(manifests) != 1:\n        raise ValueError(\'Attach exactly one successfully saved test-preparation output\')\n    root = manifests[0].parent\n    frame, split_hash = read_frozen_test_inputs(root, training_directory, freeze)\n    manifest = json.loads(manifests[0].read_text())\n    if (manifest.get(\'complete\') is not True or manifest[\'freeze_digest\'] != json_digest(freeze)\n            or manifest[\'test_csv_sha256\'] != split_hash or len(manifest[\'entries\']) != len(frame)):\n        raise ValueError(\'Test cache is incomplete or differs from the pre-test freeze/split\')\n    identity = manifest[\'preprocessing_identity\']\n    if (identity[\'preprocessing\'] != freeze[\'preprocessing\']\n            or identity[\'implementation\'] != freeze[\'training_implementation_sha256\'][\'unetpp_baseline\']):\n        raise ValueError(\'Test preprocessing differs from the frozen training method\')\n    paths=[]\n    for index, row in enumerate(frame.itertuples(index=False)):\n        entry=manifest[\'entries\'][index]\n        expected={\'signature\':json_digest(identity), \'clip_id\':row.clip_id, \'file_path\':row.file_path,\n            \'source_sha256\':entry[\'source\'][\'sha256\'], \'frame_count\':int(row.frame_count),\n            \'height\':int(row.height), \'width\':int(row.width), \'target\':[96,128,128]}\n        if (entry[\'index\'] != index or entry[\'clip_id\'] != row.clip_id or entry[\'label\'] != int(row.label)\n                or entry[\'check\'][\'identity\'] != expected):\n            raise ValueError(\'Test cache identity/order/source provenance differs\')\n        relative=Path(entry[\'array\'])\n        if relative.is_absolute() or \'..\' in relative.parts:\n            raise ValueError(\'Unsafe test cache path\')\n        path=root/relative\n        if file_digest(path) != entry[\'check\'][\'tensor_sha256\']:\n            raise ValueError(f\'Test tensor checksum differs: {row.clip_id}\')\n        array=np.load(path,mmap_mode=\'r\',allow_pickle=False)\n        if array.dtype != np.float32 or array.shape != (1,96,128,128):\n            raise ValueError(\'Test tensor shape/dtype differs\')\n        paths.append(path)\n        if (index+1)%100==0 or index+1==len(frame):\n            report(f\'Verified test input {index+1}/{len(frame)}\')\n    return frame, CachedClips(paths,frame[\'label\']), manifest\n\n\ndef evaluate_frozen_model(model_factory, dataset, frame, training_directory, freeze,\n                          output_directory, runtime, device=\'cuda\', report=print):\n    """One forward pass per selected test clip, with a fixed threshold and no optimizer."""\n    verify_frozen_training(training_directory,freeze)\n    if len(dataset) != len(frame) or len(frame) != freeze[\'test_clips\']:\n        raise ValueError(\'Test dataset/manifest count differs from freeze\')\n    output_directory=Path(output_directory);output_directory.mkdir(parents=True,exist_ok=True)\n    if (output_directory/\'test_metrics.json\').exists():\n        raise ValueError(\'Test results already exist; inspect them rather than rerunning model selection\')\n    payload=torch.load(Path(training_directory)/\'best.pt\',map_location=\'cpu\',weights_only=True)\n    if (payload[\'epoch\'] != freeze[\'selected_epoch\'] or json_digest(payload[\'config\']) != freeze[\'training_config_digest\']\n            or payload[\'data_fingerprint\'] != freeze[\'training_data_fingerprint\']):\n        raise ValueError(\'Checkpoint payload differs from frozen selection\')\n    device=torch.device(device)\n    model=model_factory().to(device);model.load_state_dict(payload[\'model_state\']);model.eval()\n    before={name:value.detach().cpu().clone() for name,value in model.state_dict().items()}\n    generator=torch.Generator().manual_seed(0)\n    loader=DataLoader(dataset,batch_size=freeze[\'physical_batch\'],shuffle=False,num_workers=0,\n        pin_memory=device.type==\'cuda\',generator=generator)\n    probabilities=[];labels=[];loss_sum=0.;start=time.perf_counter()\n    criterion=torch.nn.BCEWithLogitsLoss()\n    with torch.inference_mode():\n        for inputs,target in loader:\n            inputs,target=inputs.to(device),target.to(device)\n            with autocast_for(device,freeze[\'precision\']):\n                logits=model(inputs).reshape(-1);loss=criterion(logits,target)\n            if not torch.isfinite(loss):raise ValueError(\'Nonfinite test inference loss\')\n            probabilities.extend(torch.sigmoid(logits.float()).cpu().tolist());labels.extend(target.cpu().tolist())\n            loss_sum+=loss.item()*len(target)\n            if len(labels)%100==0 or len(labels)==len(frame):report(f\'Inferred test clips {len(labels)}/{len(frame)}\')\n    if not np.array_equal(np.asarray(labels),frame[\'label\'].to_numpy()):\n        raise ValueError(\'Inference label order differs from official test manifest\')\n    if any(not torch.equal(value.detach().cpu(),before[name]) for name,value in model.state_dict().items()):\n        raise ValueError(\'Model state changed during test evaluation\')\n    metrics=binary_metrics(np.asarray(labels),np.asarray(probabilities),freeze[\'threshold\'])\n    prediction=frame[[\'clip_id\',\'file_path\',\'label\']].copy()\n    prediction[\'rd_probability\']=probabilities\n    prediction[\'predicted_label\']=(prediction[\'rd_probability\']>=freeze[\'threshold\']).astype(int)\n    prediction.to_csv(output_directory/\'test_predictions.csv\',index=False)\n    frame.to_csv(output_directory/\'test_manifest.csv\',index=False)\n    result={\'evaluation_id\':freeze[\'evaluation_id\'],\'partition\':\'official_test\',\'test_clips\':len(frame),\n        \'selected_epoch\':freeze[\'selected_epoch\'],\'checkpoint_sha256\':freeze[\'checkpoint_sha256\'],\n        \'freeze_digest\':json_digest(freeze),\'precision\':freeze[\'precision\'],\'runtime\':runtime,\n        \'loss\':loss_sum/len(frame),\'seconds\':time.perf_counter()-start,\'metrics\':metrics,\n        \'test_used\':True,\'training_performed\':False,\'threshold_tuned\':False}\n    write_json_atomic(output_directory/\'test_metrics.json\',result)\n    write_json_atomic(output_directory/\'evaluation_freeze.json\',freeze)\n    reference=freeze[\'published_reference\']\n    pd.DataFrame([{\'source\':\'our frozen UNet++\',\'partition\':\'official_test\',**metrics},\n                  {\'source\':\'ERDES article Table 8 (author reported)\',\'partition\':\'test\',\n                   **{key:reference[key] for key in [\'sensitivity\',\'specificity\',\'precision\',\'F1\',\'accuracy\']}}]).to_csv(\n        output_directory/\'test_reference_comparison.csv\',index=False)\n    return result\n'
r2p1_load_module('unetpp_evaluation', R2P1_MODULE_SOURCE)


In [ ]:
R2P1_MODULE_SOURCE = '"""Frozen-checkpoint evaluation of comparison models on the reused official test split.\n\nModel-agnostic (R(2+1)D-18, R3D-18). It reuses the completed U-Net++ test-input\ncache without relabeling its freeze: the cache\'s own freeze must hash to its\nmanifest digest and to a pinned digest, the official split archive is re-checked,\nand every tensor checksum, label, order and preprocessing identity is verified.\nThe training artifact is verified before the test CSV is opened. No optimizer,\nthreshold search, epoch selection, calibration or retraining exists here.\n"""\nimport hashlib\nimport io\nimport json\nfrom pathlib import Path\nimport time\nimport zipfile\n\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import DataLoader\n\nfrom rd_ultrasound_ml.unetpp_data import CachedClips\nfrom rd_ultrasound_ml.unetpp_evaluation import validate_test_partition\nfrom rd_ultrasound_ml.unetpp_kaggle import read_official_inputs\nfrom rd_ultrasound_ml.unetpp_training import (\n    autocast_for, binary_metrics, file_digest, json_digest, write_json_atomic,\n)\n\nBENCHMARK_LABEL = \'reused official test benchmark\'\n\n\ndef verify_comparison_training(directory, freeze):\n    """Check the team freeze and the selected training artifact; must precede any test access."""\n    directory = Path(directory)\n    if freeze.get(\'team_freeze_confirmed\') is not True:\n        raise ValueError(\'Team has not confirmed the frozen model choices; set team_freeze_confirmed \'\n                         \'in the evaluation config in a reviewed commit before any test access\')\n    if (freeze[\'training_epochs_completed\'] != 50 or freeze[\'threshold\'] != .5\n            or freeze[\'test_accessed_at_freeze\'] is not False or freeze[\'checkpoint_file\'] != \'best.pt\'):\n        raise ValueError(\'Invalid pre-test freeze: require complete training and the fixed protocol\')\n    config = json.loads((directory/\'config.json\').read_text())\n    summary = json.loads((directory/\'summary.json\').read_text())\n    if (config.get(\'experiment_id\') != freeze[\'training_experiment_id\']\n            or config.get(\'implementation_sha256\') != freeze[\'training_implementation_sha256\']):\n        raise ValueError(\'Training experiment or implementation hashes differ from freeze\')\n    if (json_digest(config) != freeze[\'training_config_digest\']\n            or summary[\'config_digest\'] != freeze[\'training_config_digest\']\n            or summary[\'data_fingerprint\'] != freeze[\'training_run_fingerprint\']):\n        raise ValueError(\'Frozen training configuration/run fingerprint differs\')\n    if (summary[\'status\'] != \'completed\' or summary[\'epochs_completed\'] != 50\n            or summary[\'best_epoch\'] != freeze[\'selected_epoch\'] or summary[\'test_used\']):\n        raise ValueError(\'Training is incomplete or selected epoch differs from freeze\')\n    identity = json.loads((directory/\'input_fingerprint.json\').read_text())[\'preprocessing_identity\']\n    if identity != freeze[\'preprocessing_identity\']:\n        raise ValueError(\'Training preprocessing identity differs from freeze\')\n    if file_digest(directory/freeze[\'checkpoint_file\']) != freeze[\'checkpoint_sha256\']:\n        raise ValueError(\'Selected checkpoint SHA256 differs from pre-test freeze\')\n    return config\n\n\ndef find_comparison_training(input_root, freeze):\n    matches = [path.parent for path in Path(input_root).rglob(\'best.pt\')\n               if file_digest(path) == freeze[\'checkpoint_sha256\']]\n    if len(matches) != 1:\n        raise ValueError(\'Attach exactly one matching final-training output for the frozen checkpoint\')\n    verify_comparison_training(matches[0], freeze)\n    return matches[0]\n\n\ndef load_reused_test_cache(input_root, training_directory, freeze, report=print):\n    """Verify the reused test-input cache under this model\'s freeze; return frame, dataset, record."""\n    verify_comparison_training(training_directory, freeze)  # Must precede opening test.csv.\n    manifests = list(Path(input_root).rglob(\'test_prepared_manifest.json\'))\n    if len(manifests) != 1:\n        raise ValueError(\'Attach exactly one successfully saved test-preparation output\')\n    root = manifests[0].parent\n    manifest = json.loads(manifests[0].read_text())\n    preparation_freeze = json.loads((root/\'evaluation_freeze.json\').read_text())\n    pinned = freeze[\'test_cache\'][\'preparation_freeze_digest\']\n    if (manifest.get(\'complete\') is not True or json_digest(preparation_freeze) != manifest[\'freeze_digest\']\n            or manifest[\'freeze_digest\'] != pinned):\n        raise ValueError(\'Test cache is incomplete or its preparation freeze differs from the pin\')\n    train, validation = read_official_inputs(root)  # Checks official archive/metadata MD5s.\n    with zipfile.ZipFile(root/\'splits.zip\') as archive:\n        content = archive.read(\'splits/non_rd_vs_rd/test.csv\')\n    split_hash = hashlib.sha256(content).hexdigest()\n    metadata = pd.read_csv(root/\'erdes_metadata.csv\')\n    frame = validate_test_partition(pd.read_csv(io.BytesIO(content)), metadata, train, validation)\n    if manifest[\'test_csv_sha256\'] != split_hash or len(manifest[\'entries\']) != len(frame):\n        raise ValueError(\'Test cache differs from the official test split\')\n    if len(frame) != freeze[\'test_clips\'] or frame[\'label\'].value_counts().to_dict() != {\n            int(key): value for key, value in freeze[\'test_label_counts\'].items()}:\n        raise ValueError(\'Official test count/labels differ from freeze\')\n    identity = manifest[\'preprocessing_identity\']\n    if identity != freeze[\'preprocessing_identity\']:\n        raise ValueError(\'Test preprocessing identity differs from the training inputs\')\n    signature = json_digest(identity)\n    paths = []\n    for index, row in enumerate(frame.itertuples(index=False)):\n        entry = manifest[\'entries\'][index]\n        expected = {\'signature\': signature, \'clip_id\': row.clip_id, \'file_path\': row.file_path,\n            \'source_sha256\': entry[\'source\'][\'sha256\'], \'frame_count\': int(row.frame_count),\n            \'height\': int(row.height), \'width\': int(row.width), \'target\': [96, 128, 128]}\n        if (entry[\'index\'] != index or entry[\'clip_id\'] != row.clip_id or entry[\'label\'] != int(row.label)\n                or entry[\'check\'][\'identity\'] != expected):\n            raise ValueError(\'Test cache identity/order/source provenance differs\')\n        relative = Path(entry[\'array\'])\n        if relative.is_absolute() or \'..\' in relative.parts:\n            raise ValueError(\'Unsafe test cache path\')\n        path = root/relative\n        if file_digest(path) != entry[\'check\'][\'tensor_sha256\']:\n            raise ValueError(f\'Test tensor checksum differs: {row.clip_id}\')\n        array = np.load(path, mmap_mode=\'r\', allow_pickle=False)\n        if array.dtype != np.float32 or array.shape != (1, 96, 128, 128):\n            raise ValueError(\'Test tensor shape/dtype differs\')\n        paths.append(path)\n        if (index+1) % 100 == 0 or index+1 == len(frame):\n            report(f\'Verified test input {index+1}/{len(frame)}\')\n    record = {\'freeze_digest\': json_digest(freeze), \'preparation_freeze_digest\': pinned,\n        \'test_csv_sha256\': split_hash, \'prepared_manifest_sha256\': json_digest(manifest),\n        \'preprocessing_identity\': identity, \'test_clips\': len(frame)}\n    return frame, CachedClips(paths, frame[\'label\']), record\n\n\ndef evaluate_comparison_model(model_factory, dataset, frame, training_directory, freeze,\n                              output_directory, runtime, device=\'cuda\', report=print):\n    """One forward pass per test clip at the frozen threshold; records inference timing."""\n    verify_comparison_training(training_directory, freeze)\n    if len(dataset) != len(frame) or len(frame) != freeze[\'test_clips\']:\n        raise ValueError(\'Test dataset/manifest count differs from freeze\')\n    output_directory = Path(output_directory)\n    output_directory.mkdir(parents=True, exist_ok=True)\n    if (output_directory/\'test_metrics.json\').exists():\n        raise ValueError(\'Test results already exist; inspect them rather than rerunning\')\n    payload = torch.load(Path(training_directory)/freeze[\'checkpoint_file\'], map_location=\'cpu\', weights_only=True)\n    if (payload[\'epoch\'] != freeze[\'selected_epoch\'] or json_digest(payload[\'config\']) != freeze[\'training_config_digest\']\n            or payload[\'data_fingerprint\'] != freeze[\'training_run_fingerprint\'] or payload.get(\'test_used\')):\n        raise ValueError(\'Checkpoint payload differs from frozen selection\')\n    device = torch.device(device)\n    model = model_factory().to(device)\n    model.load_state_dict(payload[\'model_state\'])\n    model.eval()\n    before = {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}\n    loader = DataLoader(dataset, batch_size=freeze[\'physical_batch\'], shuffle=False, num_workers=0,\n                        pin_memory=device.type == \'cuda\')\n    probabilities, labels, forward_seconds, loss_sum = [], [], [], 0.\n    criterion = torch.nn.BCEWithLogitsLoss()\n    synchronize = torch.cuda.synchronize if device.type == \'cuda\' else (lambda: None)\n    start = time.perf_counter()\n    with torch.inference_mode():\n        for inputs, target in loader:\n            inputs, target = inputs.to(device), target.to(device)\n            synchronize()\n            began = time.perf_counter()\n            with autocast_for(device, freeze[\'precision\']):\n                logits = model(inputs).reshape(-1)\n            synchronize()\n            forward_seconds.append(time.perf_counter()-began)\n            loss = criterion(logits.float(), target)\n            if not torch.isfinite(loss):\n                raise ValueError(\'Nonfinite test inference loss\')\n            probabilities.extend(torch.sigmoid(logits.float()).cpu().tolist())\n            labels.extend(target.cpu().tolist())\n            loss_sum += loss.item()*len(target)\n            if len(labels) % 100 == 0 or len(labels) == len(frame):\n                report(f\'Inferred test clips {len(labels)}/{len(frame)}\')\n    total_seconds = time.perf_counter()-start\n    if not np.array_equal(np.asarray(labels), frame[\'label\'].to_numpy()):\n        raise ValueError(\'Inference label order differs from official test manifest\')\n    if any(not torch.equal(value.detach().cpu(), before[name]) for name, value in model.state_dict().items()):\n        raise ValueError(\'Model state changed during test evaluation\')\n    metrics = binary_metrics(np.asarray(labels), np.asarray(probabilities), freeze[\'threshold\'])\n    prediction = frame[[\'clip_id\', \'file_path\', \'label\']].copy()\n    prediction[\'rd_probability\'] = probabilities\n    prediction[\'predicted_label\'] = (prediction[\'rd_probability\'] >= freeze[\'threshold\']).astype(int)\n    prediction.to_csv(output_directory/\'test_predictions.csv\', index=False)\n    frame.to_csv(output_directory/\'test_manifest.csv\', index=False)\n    first_clips = min(freeze[\'physical_batch\'], len(frame))\n    steady_seconds, steady_clips = ((sum(forward_seconds[1:]), len(frame)-first_clips) if len(forward_seconds) > 1\n                                    else (sum(forward_seconds), len(frame)))\n    timing = {\'hardware\': runtime.get(\'gpu\') or str(device), \'physical_batch\': freeze[\'physical_batch\'],\n        \'precision\': freeze[\'precision\'], \'clips\': len(frame),\n        \'seconds_including_disk_loading\': total_seconds,\n        \'model_forward_seconds\': sum(forward_seconds), \'first_batch_seconds\': forward_seconds[0],\n        \'model_forward_ms_per_clip_excluding_first_batch\': 1000*steady_seconds/steady_clips}\n    result = {\'evaluation_id\': freeze[\'evaluation_id\'], \'model\': freeze[\'model_label\'],\n        \'partition\': \'official_test\', \'benchmark_label\': BENCHMARK_LABEL, \'test_clips\': len(frame),\n        \'selected_epoch\': freeze[\'selected_epoch\'], \'checkpoint_sha256\': freeze[\'checkpoint_sha256\'],\n        \'freeze_digest\': json_digest(freeze), \'precision\': freeze[\'precision\'], \'runtime\': runtime,\n        \'loss\': loss_sum/len(frame), \'inference_timing\': timing, \'metrics\': metrics,\n        \'test_used\': True, \'training_performed\': False, \'threshold_tuned\': False}\n    write_json_atomic(output_directory/\'test_metrics.json\', result)\n    write_json_atomic(output_directory/\'evaluation_freeze.json\', freeze)\n    rows = [{\'source\': freeze[\'model_label\'] + \' (this evaluation)\', \'partition\': BENCHMARK_LABEL, **metrics}]\n    rows += [dict(row) for row in freeze.get(\'reference_test_rows\', [])]\n    pd.DataFrame(rows).to_csv(output_directory/\'test_reference_comparison.csv\', index=False)\n    return result\n'
r2p1_load_module('comparison_evaluation', R2P1_MODULE_SOURCE)


In [ ]:
from rd_ultrasound_ml.r2plus1d18_baseline import R2Plus1D18RDClassifier
from rd_ultrasound_ml.unetpp_training import file_digest, json_digest, write_json_atomic
from rd_ultrasound_ml.comparison_evaluation import (find_comparison_training, load_reused_test_cache,
    evaluate_comparison_model)
if R2P1_FREEZE.get('team_freeze_confirmed') is not True:
    raise RuntimeError('Team freeze not confirmed in the evaluation config; no test data has been opened.')
for name, digest in R2P1_FREEZE['training_implementation_sha256'].items():
    if hashlib.sha256(r2p1_sources_embedded[name].encode()).hexdigest() != digest:
        raise ValueError('Embedded training implementation differs from freeze: ' + name)
r2p1_training_dir = find_comparison_training(Path('/kaggle/input'), R2P1_FREEZE)
print('Frozen training verified: epoch', R2P1_FREEZE['selected_epoch'], '| checkpoint SHA256:',
      R2P1_FREEZE['checkpoint_sha256'])


In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('Select an available Kaggle CUDA GPU for frozen test inference.')
if r2p1_runtime != R2P1_FREEZE['training_runtime'] or json.loads((r2p1_training_dir/'runtime.json').read_text()) != r2p1_runtime:
    raise ValueError('GPU runtime differs from frozen training; no test data opened. Inspect and document before inference.')
r2p1_test, r2p1_test_dataset, r2p1_test_record = load_reused_test_cache(
    Path('/kaggle/input'), r2p1_training_dir, R2P1_FREEZE, report=lambda value: print(value, flush=True))
r2p1_output = Path('/kaggle/working') / R2P1_FREEZE['evaluation_id']
r2p1_output.mkdir(parents=True, exist_ok=True)
write_json_atomic(r2p1_output/'runtime.json', r2p1_runtime)
write_json_atomic(r2p1_output/'test_input_provenance.json', r2p1_test_record)
for name, source in r2p1_sources_embedded.items():
    (r2p1_output/(name+'.py')).write_text(source)
print('READY for frozen inference:', len(r2p1_test_dataset), 'official test clips.')


In [ ]:
r2p1_result = evaluate_comparison_model(R2Plus1D18RDClassifier, r2p1_test_dataset, r2p1_test,
    r2p1_training_dir, R2P1_FREEZE, r2p1_output, r2p1_runtime,
    device='cuda', report=lambda value: print(value, flush=True))
print('REUSED OFFICIAL TEST METRICS:', r2p1_result['metrics'])
print('Inference timing:', r2p1_result['inference_timing'])
display(pd.read_csv(r2p1_output/'test_reference_comparison.csv'))


In [ ]:
r2p1_hashes = {path.name: file_digest(path) for path in sorted(r2p1_output.iterdir())
    if path.is_file() and path.suffix in {'.json', '.csv', '.py'} and path.name != 'artifact_sha256.json'}
write_json_atomic(r2p1_output/'artifact_sha256.json', r2p1_hashes)
r2p1_zip = r2p1_output.parent/(R2P1_FREEZE['evaluation_id'] + '_results.zip')
with zipfile.ZipFile(r2p1_zip, 'w', zipfile.ZIP_DEFLATED) as archive:
    for name in [*r2p1_hashes, 'artifact_sha256.json']:
        archive.write(r2p1_output/name, arcname=f'{r2p1_output.name}/{name}')
print('Saved test results ZIP:', r2p1_zip)
print('Single frozen evaluation complete. No model was trained or tuned in this notebook.')
